# Ridge Decoder Subset Performance — Grid 1 vs Grid 2

Loads `ridge_decoder_neuron_subsets_motor_cut_grid1.parquet` and `ridge_decoder_neuron_subsets_motor_cut_grid2.parquet` for all V1 motor sessions and plots decoder performance (Mean R²) vs subset size, colour-coded by nominal imaging depth.

**Grid 1** — balanced high-OF conditions (OF ∈ {64, 256, 1024}).

**Grid 2** — balanced low-OF conditions (OF ∈ {1, 4, 16, 64}).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import flexiznam as flz
import pandas as pd
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path

from cottage_analysis.summary_analysis import load_project_subsets

# Style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'

SESSIONS_TO_EXCLUDE = {
    "PZAG22.1b_S20260220": "1000 more frames than triggers in the treadmill recording"
}


In [ ]:
PROJECTS = ["ccyp_l5_3d_vision", "colasa_3d-vision_revisions"]
GRIDS = ["grid1", "grid2"]

all_data = {}
for grid_name in GRIDS:
    filename = f"ridge_decoder_neuron_subsets_motor_cut_{grid_name}.parquet"
    dfs = []
    for project in PROJECTS:
        df = load_project_subsets(
            project,
            filename=filename,
            session_to_exclude=SESSIONS_TO_EXCLUDE.keys(),
            cut_treadmill=True,
        )
        df["project"] = project
        dfs.append(df)
    all_data[grid_name] = pd.concat(dfs, ignore_index=True)
    print(f"Grid {grid_name}: {len(all_data[grid_name])} rows, "
          f"{all_data[grid_name]['session'].nunique()} sessions")

print("\nTargets:", all_data['grid1']['target'].unique())
print("Conditions:", all_data['grid1']['condition'].unique())


In [ ]:
def depth_colormap(depths):
    """Return a dict {depth: colour} using a sequential colourmap."""
    depths_sorted = np.sort(np.unique(depths))
    norm = mcolors.Normalize(vmin=depths_sorted.min(), vmax=depths_sorted.max())
    cmap = cm.viridis
    return {d: cmap(norm(d)) for d in depths_sorted}


In [ ]:
def plot_subsets_by_depth(ax, df, target, condition="closedloop", title=""):
    """Plot R² vs subset size lines, one per session, coloured by nominal depth."""
    sub = df[(df["target"] == target) & (df["condition"] == condition)]
    if sub.empty:
        ax.set_title(title + " (no data)")
        return

    depths = sub["nominal_depth"].dropna().values
    color_map = depth_colormap(depths)

    for session, sdf in sub.groupby("session"):
        depth = sdf["nominal_depth"].iloc[0]
        color = color_map.get(depth, "grey")
        sdf_sorted = sdf.sort_values("subset_size")
        ax.plot(sdf_sorted["subset_size"], sdf_sorted["r2_mean"],
                color=color, alpha=0.7, linewidth=1.5)

    ax.set_xlabel("Subset size (# neurons)")
    ax.set_ylabel("Mean R²")
    ax.set_title(title)
    ax.set_ylim(bottom=0)

    # colourbar
    depths_all = sub["nominal_depth"].dropna().unique()
    norm = mcolors.Normalize(vmin=np.min(depths_all), vmax=np.max(depths_all))
    sm = cm.ScalarMappable(cmap=cm.viridis, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label="Nominal depth (µm)")


In [ ]:
target = "depth"
condition = "closedloop"

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, grid_name in zip(axes, GRIDS):
    df = all_data[grid_name]
    n_sess = df['session'].nunique()
    plot_subsets_by_depth(
        ax, df, target=target, condition=condition,
        title=f"{grid_name} — decode depth ({n_sess} sessions)"
    )

fig.suptitle(f"Decode {target} from neuronal subsets ({condition})", fontsize=14)
fig.tight_layout()


In [ ]:
target = "OF_stim"

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, grid_name in zip(axes, GRIDS):
    df = all_data[grid_name]
    n_sess = df['session'].nunique()
    plot_subsets_by_depth(
        ax, df, target=target, condition="closedloop",
        title=f"{grid_name} — decode OF ({n_sess} sessions)"
    )

fig.suptitle("Decode optic flow from neuronal subsets (closedloop)", fontsize=14)
fig.tight_layout()


In [ ]:
target = "RS_stim"

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, grid_name in zip(axes, GRIDS):
    df = all_data[grid_name]
    n_sess = df['session'].nunique()
    plot_subsets_by_depth(
        ax, df, target=target, condition="closedloop",
        title=f"{grid_name} — decode RS ({n_sess} sessions)"
    )

fig.suptitle("Decode running speed from neuronal subsets (closedloop)", fontsize=14)
fig.tight_layout()


In [ ]:
# Quick numeric summary: mean R² at largest subset size per session
rows = []
for grid_name, df in all_data.items():
    for target in ["OF_stim", "RS_stim", "depth"]:
        sub = df[(df["target"] == target) & (df["condition"] == "closedloop")]
        if sub.empty:
            continue
        max_r2 = sub.groupby("session")["r2_mean"].max()
        rows.append({
            "grid": grid_name,
            "target": target,
            "mean_max_r2": max_r2.mean(),
            "sem_max_r2": max_r2.sem(),
            "n_sessions": len(max_r2),
        })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))


In [ ]:
# Separate by layers
cut_off = 350


target_mapping = {
    "OF_stim": "Optic Flow (OF)",
    "RS_stim": "Running Speed (RS)",
    "depth": "Depth"
}

# Optional color mapping to keep targets color-consistent
colors = {
    "OF_stim": "C0",
    "RS_stim": "C1",
    "depth": "C2"
}

cond = "closedloop"
for grid_name, df in all_data.items():
    max_sizes = df.groupby("session")["subset_size"].transform("max")
    filtered_subsets_df = df[df["subset_size"] < max_sizes]

    fig, ax = plt.subplots(figsize=(10, 7))
    for target, title in target_mapping.items():
        cond_df = filtered_subsets_df[(filtered_subsets_df["target"] == target) & (filtered_subsets_df["condition"] == cond)]
        color = colors[target]
        
        # --- Layer 2/3 (depth <= cut_off) -> Circles ---
        layer23 = cond_df[cond_df.nominal_depth <= cut_off]
        if not layer23.empty:
            avg_l23 = layer23.groupby('subset_size')['r2_mean'].mean()
            ax.plot(avg_l23.index, avg_l23.values, label=f"{title} (L2/3)", marker='o', color=color, linewidth=2)
            
        # --- Layer 5 (depth > cut_off) -> Squares ---
        layer5 = cond_df[cond_df.nominal_depth > cut_off]
        if not layer5.empty:
            avg_l5 = layer5.groupby('subset_size')['r2_mean'].mean()
            ax.plot(avg_l5.index, avg_l5.values, label=f"{title} (L5)", marker='s', color=color, linewidth=2, linestyle='--')

    # --- Label axis and clean up styling ---
    ax.set_title(rf"Decoder Performance by Layer (Cut-off: {cut_off} µm)")
    ax.set_xlabel("Subset Size")
    ax.set_ylabel("Mean $R^2$")
    ax.set_xscale("log")  # Using log scale since subset sizes typically scale exponentially
    ax.legend(frameon=False, bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()


In [ ]:
# Plot the 2 grids that we know what is 1 and what is two
col = dict(grid1='dodgerblue', grid2='tomato')
for grid_name, df in all_data.items():
    conds = []
    for cond, _ in df.groupby(['OF_stim', 'RS_stim']):
        conds.append(cond)
    conds =np.vstack(conds)
    plt.plot(conds[:,0], cond[:,1], color=col[grid_name], label=grid_name)
plt.legend(loc='lower right')
plt.set_aspect('equal')


In [ ]:
df['condition']